# 1. Introduction

The previous notebook validated the structure and completeness of the movement metadata contained in the Parkinson's Disease Smartwatch Dataset (PADS). Once the metadata quality was confirmed, the next step is to inspect the raw inertial recordings collected from the smartwatch sensors.

Each movement recording consists of a time channel together with six inertial sensor channels corresponding to the three-axis accelerometer and three-axis gyroscope measurements. According to the official preprocessing scripts provided with the dataset, these recordings are used as the input for all subsequent preprocessing and machine learning analyses.

The purpose of this notebook is to examine the raw signals before any preprocessing is applied. Specifically, the notebook verifies the recording structure, timestamp consistency, sampling frequency, recording duration, and signal quality, while also comparing recordings acquired from the left and right wrists. These exploratory analyses ensure that the raw data satisfy the requirements for reliable feature extraction and predictive modeling.

## Objectives

This notebook aims to:

- Load representative raw movement recordings from the PADS dataset.
- Inspect the structure of the raw signal files.
- Verify timestamp consistency and estimate the effective sampling frequency.
- Validate the recording duration and number of samples.
- Compare left- and right-wrist recordings.
- Visualize representative accelerometer and gyroscope signals.
- Assess the quality of the raw recordings before preprocessing and feature extraction.

In [1]:
# =============================================================================
# Libraries
# =============================================================================

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path

# 2. Loading Raw Signal Data

To inspect the raw movement recordings, a representative example is selected from the PADS dataset. Each recording contains one time channel together with six inertial sensor channels corresponding to the three-axis accelerometer and three-axis gyroscope measurements.

The signals are loaded into a Pandas DataFrame to facilitate inspection, visualization, and quality assessment throughout the remainder of this notebook.

In [2]:
#Define paths
movement_path = Path("../data/raw/movement/timeseries")

print(f"Movement directory: {movement_path.resolve()}")
print(f"Directory exists: {movement_path.exists()}")

Movement directory: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\raw\movement\timeseries
Directory exists: True


In [3]:
#Locate an Example Recording
movement_files = sorted(movement_path.rglob("*.txt"))

print(f"Number of raw signal files: {len(movement_files)}")
print()

print("First five recordings:")
for file in movement_files[:5]:
    print(file.name)

Number of raw signal files: 10318

First five recordings:
001_CrossArms_LeftWrist.txt
001_CrossArms_RightWrist.txt
001_DrinkGlas_LeftWrist.txt
001_DrinkGlas_RightWrist.txt
001_Entrainment_LeftWrist.txt


In [4]:
#load an example record
columns = [
    "time",
    "acc_x",
    "acc_y",
    "acc_z",
    "gyro_x",
    "gyro_y",
    "gyro_z"
]

example_file = movement_files[0]

signal = pd.read_csv(
    example_file,
    header=None,
    names=columns
)

print(f"Example recording: {example_file.name}")
signal.head()

Example recording: 001_CrossArms_LeftWrist.txt


,time,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,0.000000,0.003235,0.012430,0.005110,-0.017059,-0.023757,-0.000578
1,0.009805,0.002275,0.013380,0.005852,-0.014924,-0.025833,0.004746
2,0.019807,0.001332,0.016225,0.005543,-0.009529,-0.033264,0.005853
3,0.029791,0.001354,0.018107,0.007207,-0.007440,-0.029995,0.009008
4,0.039801,0.001441,0.016004,0.004999,-0.011757,-0.027815,0.017495


# 3. Raw Signal Structure Inspection

Before performing any signal analysis, the structure of the raw recording is inspected to verify that the data have been loaded correctly. This section examines the dimensions of the recording, data types, descriptive statistics, and channel organization. These checks ensure that the raw signals conform to the expected format before evaluating their temporal characteristics and signal quality.

In [5]:
# 3.1 Dataset dimensions
print(f"Recording shape: {signal.shape}")

rows, cols = signal.shape

summary = pd.DataFrame({
    "Property": [
        "Number of samples",
        "Number of channels"
    ],
    "Value": [
        rows,
        cols
    ]
})

summary

Recording shape: (1024, 7)


,Property,Value
0,Number of samples,1024
1,Number of channels,7


In [6]:
#3.2. Preview of the signal
signal.head()

,time,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,0.000000,0.003235,0.012430,0.005110,-0.017059,-0.023757,-0.000578
1,0.009805,0.002275,0.013380,0.005852,-0.014924,-0.025833,0.004746
2,0.019807,0.001332,0.016225,0.005543,-0.009529,-0.033264,0.005853
3,0.029791,0.001354,0.018107,0.007207,-0.007440,-0.029995,0.009008
4,0.039801,0.001441,0.016004,0.004999,-0.011757,-0.027815,0.017495


In [7]:
#3.3 dataset information
signal.info()

<class 'pandas.DataFrame'>
RangeIndex: 1024 entries, 0 to 1023
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   time    1024 non-null   float64
 1   acc_x   1024 non-null   float64
 2   acc_y   1024 non-null   float64
 3   acc_z   1024 non-null   float64
 4   gyro_x  1024 non-null   float64
 5   gyro_y  1024 non-null   float64
 6   gyro_z  1024 non-null   float64
dtypes: float64(7)
memory usage: 56.1 KB


In [8]:
#3.4. Descriptive statistics
signal.describe().T

,count,mean,std,min,25%,50%,75%,max
time,1024.0,5.112967,2.955975,0.000000,2.556150,5.112775,7.669100,10.225438
acc_x,1024.0,-0.130550,0.248221,-1.045084,-0.111962,-0.004438,0.000252,0.032630
acc_y,1024.0,-0.044359,0.222880,-0.955157,-0.007355,0.001324,0.009127,0.597029
acc_z,1024.0,0.049997,0.177653,-0.421268,-0.005614,0.000993,0.007769,0.718406
gyro_x,1024.0,-0.039097,0.579956,-2.152440,-0.043074,-0.007595,0.007677,3.150466
gyro_y,1024.0,-0.074813,0.818649,-2.372351,-0.066585,-0.007579,0.010470,3.295569
gyro_z,1024.0,-0.268944,1.905186,-6.042105,-0.021050,-0.001763,0.015443,7.007679


In [9]:
#3.5. channel description
channel_description = pd.DataFrame({
    "Channel": [
        "time",
        "acc_x",
        "acc_y",
        "acc_z",
        "gyro_x",
        "gyro_y",
        "gyro_z"
    ],
    "Description": [
        "Time",
        "Accelerometer X-axis",
        "Accelerometer Y-axis",
        "Accelerometer Z-axis",
        "Gyroscope X-axis",
        "Gyroscope Y-axis",
        "Gyroscope Z-axis"
    ]
})

channel_description

,Channel,Description
0,time,Time
1,acc_x,Accelerometer X-axis
2,acc_y,Accelerometer Y-axis
3,acc_z,Accelerometer Z-axis
4,gyro_x,Gyroscope X-axis
5,gyro_y,Gyroscope Y-axis
6,gyro_z,Gyroscope Z-axis


### Observations

The inspected recording contains **1,024 samples** and **seven numerical channels**, all stored as 64-bit floating-point values without missing observations. The first channel represents the time information, while the remaining six channels correspond to three-axis accelerometer and gyroscope measurements.

The recording duration, sampling characteristics, and temporal consistency will be examined in the following section.

# 4. Temporal Characteristics of the Recording

The temporal characteristics of the raw recording are examined to verify the consistency of the acquisition process. Since the metadata reports a nominal sampling frequency of **100 Hz**, this section estimates the effective sampling frequency directly from the recorded time values and evaluates whether the sampling interval remains stable throughout the recording.

These analyses provide evidence that the raw signals were acquired according to the expected recording protocol before any preprocessing is applied.

In [10]:
#4.1. Sampling Interval
dt = signal["time"].diff().dropna()
dt.describe()

count    1023.000000
mean        0.009996
std         0.001761
min         0.000023
25%         0.009954
50%         0.009997
75%         0.010037
max         0.042041
Name: time, dtype: float64

In [11]:
#4.2 Sampling Interval Summary
sampling_summary = pd.DataFrame({
    "Metric": [
        "Mean interval (s)",
        "Standard deviation (s)",
        "Minimum interval (s)",
        "Maximum interval (s)"
    ],
    "Value": [
        dt.mean(),
        dt.std(),
        dt.min(),
        dt.max()
    ]
})

sampling_summary

,Metric,Value
0,Mean interval (s),0.009996
1,Standard deviation (s),0.001761
2,Minimum interval (s),0.000023
3,Maximum interval (s),0.042041


In [12]:
#4.3 Estimated Sampling Frequency
estimated_fs = 1 / dt.mean()
print(f"Estimated sampling frequency: {estimated_fs:.2f} Hz")

Estimated sampling frequency: 100.04 Hz


In [13]:
#4.4 Recording Duration
duration = signal["time"].iloc[-1] - signal["time"].iloc[0]
duration_summary = pd.DataFrame({
    "Property": [
        "Recording duration (s)",
        "Number of samples",
        "Estimated sampling frequency (Hz)"
    ],
    "Value": [
        round(duration, 4),
        len(signal),
        round(estimated_fs, 2)
    ]
})

duration_summary

,Property,Value
0,Recording duration (s),10.2254
1,Number of samples,1024.0000
2,Estimated sampling frequency (Hz),100.0400


In [14]:
#4.5 Sampling Interval Distribution
fig = px.histogram(
    x=dt[(dt > 0.008) & (dt < 0.012)],
    nbins=30,
    title="Distribution of Sampling Intervals",
    labels={
        "x": "Sampling Interval (s)",
        "count": "Frequency"
    }
)

fig.update_layout(showlegend=False)

fig.show()

fig = px.line(
    x=range(len(dt)),
    y=dt,
    labels={
        "x": "Sample",
        "y": "Sampling Interval (s)"
    },
    title="Sampling Interval Across the Recording"
)

fig.show()

In [15]:
#4.6 Time Progression
fig = px.line(
    x=range(len(signal)),
    y=signal["time"],
    labels={
        "x": "Sample",
        "y": "Time (s)"
    },
    title="Time Progression Throughout the Recording"
)

fig.show()

In [16]:
#validation 
validation = pd.DataFrame({
    "Parameter": [
        "Nominal sampling frequency",
        "Estimated sampling frequency",
        "Expected interval",
        "Observed mean interval",
        "Expected duration",
        "Observed duration"
    ],
    "Expected": [
        "100 Hz",
        "-",
        "0.0100 s",
        "-",
        "10.23 s",
        "-"
    ],
    "Observed": [
        "-",
        f"{estimated_fs:.2f} Hz",
        "-",
        f"{dt.mean():.6f} s",
        "-",
        f"{duration:.4f} s"
    ]
})

validation

,Parameter,Expected,Observed
0,Nominal sampling frequency,100 Hz,-
1,Estimated sampling frequency,-,100.04 Hz
2,Expected interval,0.0100 s,-
3,Observed mean interval,-,0.009996 s
4,Expected duration,10.23 s,-
5,Observed duration,-,10.2254 s


### Observations

The temporal analysis confirms that the examined recording follows the expected acquisition protocol. The estimated sampling frequency was **100.04 Hz**, which closely matches the nominal sampling frequency of **100 Hz** reported for the PADS dataset.

The average sampling interval was **0.009996 s**, which is effectively equivalent to the expected interval of **0.0100 s**. Similarly, the observed recording duration (**10.2254 s**) is consistent with the expected duration for a recording containing **1,024 samples** acquired at approximately 100 Hz.

The distribution of sampling intervals shows that the vast majority of intervals are concentrated around **0.01 s**, indicating a stable acquisition process. Although a small number of isolated intervals deviate from the nominal value, these represent infrequent timestamp irregularities rather than systematic sampling errors, as evidenced by the overall linear time progression and the close agreement between the expected and observed recording duration.

Overall, the temporal characteristics indicate that the recording is internally consistent and suitable for subsequent preprocessing, feature extraction, and machine learning analyses.

# 5. Dataset-wide Signal Quality Assessment

While the previous sections examined a representative raw recording, this section evaluates the quality of the complete collection of raw movement recordings. Each signal file is inspected to verify its structural integrity, sampling characteristics, recording duration, and data completeness.

Performing these validation checks across the entire dataset ensures that the raw signals satisfy the quality requirements for subsequent preprocessing, feature extraction, and machine learning analyses.

In [17]:
#5.1 Analyze All Recordings
records = []
for file in movement_files:
    df = pd.read_csv(file, header=None)
    dt = df.iloc[:, 0].diff().dropna()
    records.append({
        "file": file.name,
        "samples": len(df),
        "duration_seconds": df.iloc[-1, 0] - df.iloc[0, 0],
        "sampling_frequency": 1 / dt.mean(),
        "missing_values": df.isna().sum().sum()
    })
quality_df = pd.DataFrame(records)
quality_df.head()

,file,samples,duration_seconds,sampling_frequency,missing_values
0,001_CrossArms_LeftWrist.txt,1024,10.225438,100.044613,0
1,001_CrossArms_RightWrist.txt,1024,10.298169,99.338046,0
2,001_DrinkGlas_LeftWrist.txt,1024,10.225229,100.046657,0
3,001_DrinkGlas_RightWrist.txt,1024,10.298960,99.330420,0
4,001_Entrainment_LeftWrist.txt,2048,20.460135,100.048218,0


In [18]:
#5.2 Dataset Overview
quality_df.describe().T

,count,mean,std,min,25%,50%,75%,max
samples,10318.0,1303.272727,456.072555,1024.000000,1024.000000,1024.000000,2048.000000,2048.000000
duration_seconds,10318.0,13.054098,4.571953,10.149244,10.225415,10.298176,20.456108,20.611353
sampling_frequency,10318.0,99.761245,0.434883,98.731899,99.336582,99.918110,100.048597,100.795682
missing_values,10318.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [19]:
#5.3 Missing Values
missing_summary = pd.DataFrame({
    "Metric": [
        "Files analyzed",
        "Files containing missing values"
    ],
    "Value": [
        len(quality_df),
        (quality_df["missing_values"] > 0).sum()
    ]
})

missing_summary

,Metric,Value
0,Files analyzed,10318
1,Files containing missing values,0


In [20]:
# 5.4 Recording Length Distribution
sample_counts = (
    quality_df["samples"]
    .value_counts()
    .sort_index()
    .rename_axis("Samples")
    .reset_index(name="Recordings")
)

sample_counts

,Samples,Recordings
0,1024,7504
1,2048,2814


In [21]:
fig = px.bar(
    sample_counts,
    x="Samples",
    y="Recordings",
    text="Recordings",
    title="Distribution of Recording Lengths"
)

fig.update_traces(textposition="outside")

fig.show()

In [22]:
#5.5 Recording Duration Distribution
fig = px.histogram(
    quality_df,
    x="duration_seconds",
    nbins=30,
    title="Distribution of Recording Durations"
)

fig.show()

In [23]:
#5.6 Sampling Frequency Distribution
fig = px.histogram(
    quality_df,
    x="sampling_frequency",
    nbins=25,
    title="Distribution of Estimated Sampling Frequencies"
)

fig.show()

In [24]:
#5.7 Dataset Validation Summary
validation = pd.DataFrame({
    "Validation Check": [
        "Files analyzed",
        "Missing values",
        "Minimum sampling frequency",
        "Maximum sampling frequency",
        "Minimum duration (s)",
        "Maximum duration (s)"
    ],
    "Result": [
        len(quality_df),
        quality_df["missing_values"].sum(),
        round(quality_df["sampling_frequency"].min(),2),
        round(quality_df["sampling_frequency"].max(),2),
        round(quality_df["duration_seconds"].min(),2),
        round(quality_df["duration_seconds"].max(),2)
    ]
})

validation

,Validation Check,Result
0,Files analyzed,10318.00
1,Missing values,0.00
2,Minimum sampling frequency,98.73
3,Maximum sampling frequency,100.80
4,Minimum duration (s),10.15
5,Maximum duration (s),20.61


## 5.8 Observations

The dataset-wide quality assessment confirms that all **10,318 raw movement recordings** were successfully loaded and inspected without errors. No missing values were detected in any recording, indicating complete signal acquisition across the entire dataset.

The recordings exhibit two standardized lengths: **7,504 recordings contain 1,024 samples**, while **2,814 recordings contain 2,048 samples**. These correspond to approximate recording durations of **10.2 seconds** and **20.5 seconds**, respectively, suggesting that the dataset includes two predefined recording protocols rather than inconsistently sampled signals.

The estimated sampling frequencies are tightly clustered around the nominal value of **100 Hz**, with a median frequency of **99.92 Hz** and observed values ranging from **98.73 Hz** to **100.80 Hz**. These small variations are expected due to timestamp precision and do not indicate systematic acquisition errors.

Overall, the structural consistency, absence of missing data, standardized recording lengths, and stable sampling frequencies demonstrate that the raw movement recordings satisfy the quality requirements for subsequent preprocessing, feature extraction, and machine learning analyses.


# 6. Example Signal Visualizations

After validating the structural integrity and temporal characteristics of the complete dataset, representative raw inertial signals are visualized to illustrate the measurements recorded by the smartwatch sensors.

To avoid selection bias while maintaining reproducibility, a representative recording is randomly selected from the dataset using a fixed random seed. The visualizations presented in this section provide an intuitive understanding of the accelerometer and gyroscope signals prior to any preprocessing or feature extraction.

In [25]:
#6.1 Select Representative Recording
import random

random.seed(42)

example_file = random.choice(movement_files)

signal = pd.read_csv(
    example_file,
    header=None,
    names=columns
)

print(f"Representative recording: {example_file.name}")
signal.head()

Representative recording: 083_TouchNose_LeftWrist.txt


,time,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,0.000000,0.080676,0.804709,0.129682,-0.867629,-1.267041,-2.219843
1,0.009467,0.143122,0.769328,0.164630,-0.412240,-1.342682,-2.228170
2,0.019619,0.176178,0.691302,0.211473,-0.115175,-1.370529,-2.253694
3,0.029636,0.179491,0.594609,0.254272,-0.013882,-1.331822,-2.322023
4,0.039652,0.165299,0.500407,0.293728,-0.075083,-1.253966,-2.411726


### 6.2 Accelerometer Signal Visualization

The smartwatch accelerometer measures linear acceleration along three orthogonal axes (X, Y, and Z). Visualizing these signals provides an initial understanding of the wrist motion performed during the selected task and allows a qualitative assessment of signal continuity, amplitude, and variability before preprocessing.

In [26]:
fig = px.line(
    signal,
    x="time",
    y=["acc_x", "acc_y", "acc_z"],
    title=f"Accelerometer Signals ({example_file.stem})",
    labels={
        "time": "Time (s)",
        "value": "Acceleration",
        "variable": "Axis"
    }
)

fig.update_layout(
    legend_title="Accelerometer Axis"
)

fig.show()

### Observations

The accelerometer signals exhibit smooth and continuous variations throughout the recording, indicating stable data acquisition during the execution of the **Touch Nose** task. The three axes display distinct motion patterns, reflecting the multidirectional nature of wrist movements.

A transient with relatively larger amplitudes is observed during the first second of the recording, likely corresponding to the initiation of the movement. After this initial phase, the signals oscillate within a relatively stable amplitude range without prolonged flat regions, abrupt discontinuities, or missing segments.

Overall, the recording demonstrates good signal continuity and variability, making it suitable for subsequent preprocessing and feature extraction.

### 6.3 Gyroscope Signal Visualization

The smartwatch gyroscope measures angular velocity along the three orthogonal axes (X, Y, and Z). Unlike the accelerometer, which captures linear acceleration, the gyroscope records rotational movements of the wrist during task execution.

Visualizing the gyroscope signals complements the accelerometer inspection by providing an additional perspective on the raw inertial measurements collected by the smartwatch.

In [27]:
fig = px.line(
    signal,
    x="time",
    y=["gyro_x", "gyro_y", "gyro_z"],
    title=f"Gyroscope Signals ({example_file.stem})",
    labels={
        "time": "Time (s)",
        "value": "Angular Velocity",
        "variable": "Axis"
    }
)

fig.update_layout(
    legend_title="Gyroscope Axis"
)

fig.show()

### Observations

The gyroscope signals illustrate the angular velocity recorded along the three orthogonal axes during the representative execution of the **Touch Nose** task. Compared with the accelerometer measurements, the gyroscope channels exhibit larger oscillations, reflecting the rotational movements of the wrist throughout the task.

Distinct waveform patterns are observed across the three axes, indicating that each channel captures a different component of the wrist rotation. Repeated peaks and valleys can be identified during the recording, illustrating the dynamic nature of the performed movement.

These visualizations provide a qualitative overview of the raw gyroscope measurements prior to preprocessing and feature extraction.

# 7. Left- and Right-Wrist Comparison for Short- and Long-Duration Tasks

The PADS dataset contains synchronized recordings acquired from both the left and right wrists for each movement task. Comparing these paired recordings provides an opportunity to verify that both devices follow the same acquisition protocol while illustrating the natural differences in the measured motion.

This section compares representative recordings from both wrists using the same participant and movement task.

To illustrate both recording protocols identified during the dataset-wide inspection, comparisons are presented for a representative short-duration task (Touch Nose, 1,024 samples) and a long-duration task (Entrainment, 2,048 samples).

## 7.1 Short-Duration Recording (1,024 Samples)

The comparison begins with the representative **Touch Nose** task, which follows the short-duration acquisition protocol identified during the dataset-wide inspection. This protocol produces recordings containing **1,024 samples**, corresponding to approximately **10 seconds** of data.

The left- and right-wrist recordings from the same participant are compared to verify the consistency of the acquisition protocol while illustrating the complementary motion information captured by the two smartwatch devices.

In [28]:
#7.1.1 Load Left- and Right-Wrist Recordings
left_file = example_file

right_name = example_file.name.replace("LeftWrist", "RightWrist")
right_file = example_file.parent / right_name

left_signal = pd.read_csv(
    left_file,
    header=None,
    names=columns
)

right_signal = pd.read_csv(
    right_file,
    header=None,
    names=columns
)

print(left_file.name)
print(right_file.name)

083_TouchNose_LeftWrist.txt
083_TouchNose_RightWrist.txt


In [29]:
#7.1.2 Recording Summary
comparison = pd.DataFrame({
    "Metric":[
        "Samples",
        "Duration (s)",
        "Estimated Sampling Frequency (Hz)"
    ],
    "Left Wrist":[
        len(left_signal),
        left_signal["time"].iloc[-1],
        1/left_signal["time"].diff().dropna().mean()
    ],
    "Right Wrist":[
        len(right_signal),
        right_signal["time"].iloc[-1],
        1/right_signal["time"].diff().dropna().mean()
    ]
})

comparison

,Metric,Left Wrist,Right Wrist
0,Samples,1024.000000,1024.000000
1,Duration (s),10.224112,10.297779
2,Estimated Sampling Frequency (Hz),100.057594,99.341809


In [30]:
#7.1.3 Accelerometer Comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=left_signal["time"],
        y=left_signal["acc_x"],
        name="Left Wrist",
        mode="lines"
    )
)

fig.add_trace(
    go.Scatter(
        x=right_signal["time"],
        y=right_signal["acc_x"],
        name="Right Wrist",
        mode="lines"
    )
)

fig.update_layout(
    title=f"Accelerometer X-axis Comparison ({example_file.stem.replace('_LeftWrist','')})",
    xaxis_title="Time (s)",
    yaxis_title="Acceleration"
)

fig.show()

### Observations

Both recordings contain **1,024 samples**, indicating that the left- and right-wrist devices followed the same acquisition protocol. The estimated sampling frequencies are close to the nominal value of **100 Hz**, resulting in comparable recording durations (approximately **10.2 seconds**).

Although both recordings correspond to the same participant performing the same movement task, the accelerometer signals exhibit noticeable differences in waveform and amplitude. These differences are expected because each smartwatch captures the motion of a different wrist, which naturally performs the movement with distinct kinematic patterns.

Overall, the comparison confirms that the recordings are temporally consistent while illustrating the complementary information provided by the two wrist-mounted sensors.

In [31]:
#7.1.4 Gyroscope Comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=left_signal["time"],
        y=left_signal["gyro_x"],
        name="Left Wrist",
        mode="lines"
    )
)

fig.add_trace(
    go.Scatter(
        x=right_signal["time"],
        y=right_signal["gyro_x"],
        name="Right Wrist",
        mode="lines"
    )
)

fig.update_layout(
    title=f"Gyroscope X-axis Comparison ({example_file.stem.replace('_LeftWrist','')})",
    xaxis_title="Time (s)",
    yaxis_title="Angular Velocity"
)

fig.show()

### Observations

The gyroscope recordings exhibit distinct rotational patterns for the left and right wrists during the execution of the same movement task. Prominent peaks and valleys occur at similar time points in both recordings, indicating that the two devices captured the same movement sequence.

Although the timing of the major rotational events is consistent, the waveforms and amplitudes differ between wrists. These differences are expected because the dominant and non-dominant wrists do not perform identical rotational motions during task execution.

Overall, the comparison demonstrates that both smartwatch devices recorded synchronized movements while capturing complementary information about wrist rotation.

## 7.2 Long-Duration Recording (2,048 Samples)

To complement the comparison performed on the short-duration recording, this section examines a representative long-duration task. The **Entrainment** task was selected because it follows the second acquisition protocol identified during the dataset-wide inspection, producing recordings containing **2,048 samples** (approximately 20 seconds).

Using the same participant allows the comparison to focus on the effect of the recording protocol while maintaining consistent subject characteristics.

In [32]:
#7.2.1 Load the recordings
patient = example_file.name[:3]
left_long = movement_path / f"{patient}_Entrainment_LeftWrist.txt"
right_long = movement_path / f"{patient}_Entrainment_RightWrist.txt"
left_long_signal = pd.read_csv(
    left_long,
    header=None,
    names=columns
)
right_long_signal = pd.read_csv(
    right_long,
    header=None,
    names=columns
)
print(left_long.name)
print(right_long.name)

083_Entrainment_LeftWrist.txt
083_Entrainment_RightWrist.txt


In [33]:
#7.2.2 Recording Summary
comparison_long = pd.DataFrame({
    "Metric": [
        "Samples",
        "Duration (s)",
        "Estimated Sampling Frequency (Hz)"
    ],
    "Left Wrist": [
        len(left_long_signal),
        left_long_signal["time"].iloc[-1],
        1 / left_long_signal["time"].diff().dropna().mean()
    ],
    "Right Wrist": [
        len(right_long_signal),
        right_long_signal["time"].iloc[-1],
        1 / right_long_signal["time"].diff().dropna().mean()
    ]
})

comparison_long

,Metric,Left Wrist,Right Wrist
0,Samples,2048.000000,2048.000000
1,Duration (s),20.458879,20.605322
2,Estimated Sampling Frequency (Hz),100.054356,99.343267


In [34]:
#7.2.3 Accelerometer Comparison
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=left_long_signal["time"],
        y=left_long_signal["acc_x"],
        mode="lines",
        name="Left Wrist"
    )
)
fig.add_trace(
    go.Scatter(
        x=right_long_signal["time"],
        y=right_long_signal["acc_x"],
        mode="lines",
        name="Right Wrist"
    )
)
fig.update_layout(
    title=f"Accelerometer X-axis Comparison ({patient} - Entrainment)",
    xaxis_title="Time (s)",
    yaxis_title="Acceleration"
)
fig.show()

### Observations

Both recordings contain **2,048 samples**, confirming that the left- and right-wrist devices followed the same acquisition protocol for the Entrainment task. The recording durations are approximately **20.5 seconds**, and the estimated sampling frequencies remain close to the nominal value of **100 Hz**, consistent with the results obtained for the short-duration recordings.

The accelerometer signals exhibit continuous oscillatory patterns throughout the recording, with relatively small amplitudes compared to the previously examined Touch Nose task. Although the waveforms differ between the left and right wrists, both recordings display similar temporal behavior and maintain synchronized acquisition over the entire recording period.

In [35]:
#Gyroscope Comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=left_long_signal["time"],
        y=left_long_signal["gyro_x"],
        mode="lines",
        name="Left Wrist"
    )
)

fig.add_trace(
    go.Scatter(
        x=right_long_signal["time"],
        y=right_long_signal["gyro_x"],
        mode="lines",
        name="Right Wrist"
    )
)

fig.update_layout(
    title=f"Gyroscope X-axis Comparison ({patient} - Entrainment)",
    xaxis_title="Time (s)",
    yaxis_title="Angular Velocity"
)

fig.show()

### Observations

The gyroscope recordings exhibit similar temporal behavior for both the left and right wrists throughout the **Entrainment** task. Following an initial transient at the beginning of the recording, both signals remain centered near zero with relatively small oscillations over the remainder of the acquisition period.

Compared with the previously examined **Touch Nose** task, the gyroscope signals display lower amplitudes and fewer pronounced rotational peaks, illustrating the different motion characteristics associated with this movement task. Despite these differences, both recordings preserve consistent recording durations, sample counts, and temporal alignment, further confirming the consistency of the dual-wrist acquisition protocol.

# 8. Conclusions

This notebook examined the raw smartwatch movement recordings contained in the Parkinson's Disease Smartwatch Dataset (PADS) prior to any preprocessing.

The inspection confirmed that the recordings follow a consistent acquisition protocol, with sampling frequencies centered around the nominal value of **100 Hz** and standardized recording lengths of **1,024** and **2,048 samples**, corresponding to the two recording protocols identified in the dataset. No missing values were detected across the **10,318** analyzed recordings, indicating complete signal acquisition.

Representative accelerometer and gyroscope visualizations provided an overview of the raw inertial measurements, while comparisons between left- and right-wrist recordings demonstrated that both smartwatch devices consistently captured the same movement tasks while preserving complementary motion information.

Overall, the analyses performed in this notebook confirm that the raw movement recordings are structurally consistent and suitable for the subsequent preprocessing, feature extraction, and machine learning stages of the project.